# MFA-Bachelorarbeit — Auswertung
Jehee Han, THM Gießen

Dieses Notebook lädt den aktuellen SoSci-Export, bereinigt ihn nach den
Kriterien aus der Methodik (LASTPAGE >= 14, beide Attention Checks bestanden),
kodiert die Skalen um und rechnet die Kernanalysen:

1. Deskriptive Statistik pro Verfahren (Rangfolge)
2. Gruppenvergleich technisch vs. nicht-technisch (Mann-Whitney-U + t-Test als Robustheitscheck)
3. Spearman-Korrelationen (Verständnis <-> Nutzungsbereitschaft, DF04 <-> DF06)
4. Robustheitsprüfung mit/ohne koreanische Teilstichprobe

**Um das Notebook mit einem neuen Export laufen zu lassen:** nur den Dateinamen
in Zelle 2 ändern und "Run All" drücken.

In [53]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)

## 1. Daten laden

In [54]:
# --- Hier den aktuellen Dateinamen eintragen ---
DATEINAME = "data_Bachelorarbeit_MFA_Jehee_Han_2026-10-17_03-46.csv"
PFAD = f"/mnt/user-data/uploads/{DATEINAME}"

df = pd.read_csv(DATEINAME, sep='\t', encoding='utf-16')
print(f"Rohdaten (inkl. Beschreibungszeile): {len(df)} Zeilen")

Rohdaten (inkl. Beschreibungszeile): 192 Zeilen


In [55]:
# Beschreibungszeile und eigene Testfälle entfernen
df = df[df['REF'] != 'Referenz (sofern im Link angegeben)'].copy()
df = df[df['REF'] != 'test'].copy()
print(f"Nach Bereinigung: {len(df)} echte Zugriffe")

Nach Bereinigung: 190 echte Zugriffe


## 2. Filterung nach Auswertungskriterien

Kriterium: `LASTPAGE >= 14` (vollständig durchlaufen) **und** beide Attention Checks bestanden (`AC01_03 == 2`, `AC05_02 == 5`).

In [56]:
num_cols = ['LASTPAGE','MAXPAGE','AC01_03','AC05_02','DF01','DF02','DF03','DF04','DF06',
            'MA01','MA02','MA05','MA07','TIME_SUM']
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Quelle bestimmen: Prolific-IDs sind 24-stellige Hex-Strings
df['quelle'] = np.where(df['REF'].astype(str).str.len() == 24, 'Prolific', 'Umfeld')

valid = df[df['LASTPAGE'] >= 14].copy()
print(f"Vollstaendig (LASTPAGE >= 14): {len(valid)}")

clean = valid[(valid['AC01_03'] == 2) & (valid['AC05_02'] == 5)].copy()
print(f"Beide Attention Checks bestanden: {len(clean)}")
print(f"Ausschlussquote: {(len(valid)-len(clean))/len(valid)*100:.1f}%")

Vollstaendig (LASTPAGE >= 14): 174
Beide Attention Checks bestanden: 163
Ausschlussquote: 6.3%


In [57]:
print("Quelle:")
print(clean['quelle'].value_counts())
print()
print("Sprache:")
print(clean['LANGUAGE'].value_counts())
print()
print("IT-Bezug (DF06, 1=Ja/2=Nein):")
print(clean['DF06'].value_counts())

Quelle:
quelle
Umfeld      83
Prolific    80
Name: count, dtype: int64

Sprache:
LANGUAGE
kor    73
ger    61
eng    29
Name: count, dtype: int64

IT-Bezug (DF06, 1=Ja/2=Nein):
DF06
2    95
1    68
Name: count, dtype: int64


## 3. Items umkodieren

Das Lästigkeits-Item (`_02` in jeder Batterie) ist negativ gepolt und wird für die Auswertung umgedreht (6 minus Rohwert), damit hohe Werte durchgehend "positiv" bedeuten.

In [58]:
verfahren = {
    'SMS':        'A101',
    'AuthApp':    'A201',
    'Biometrie':  'A301',
    'HWToken':    'A401',
    'FIDO2':      'A501',
}

items = {
    '01': 'einfach',      # easy to use
    '02': 'laestig_roh',  # annoying (negativ gepolt)
    '03': 'vertrauen',    # trust
    '05': 'bereitschaft', # willingness to use
    '06': 'verstehen',    # technical understanding
}

for name, prefix in verfahren.items():
    for suffix, label in items.items():
        col = f'{prefix}_{suffix}'
        clean[col] = pd.to_numeric(clean[col], errors='coerce')


# SoSci kodiert fehlende Antworten mit negativen Werten (-9, -1).
# Diese als fehlend setzen, damit sie nicht als Skalenwerte mitgerechnet werden.
alle_items = [f'{p}_{s}' for p in verfahren.values() for s in items.keys()]
alle_items += ['AC01_01','AC01_02','AC01_04','AC05_01','AC05_03',
               'DF01','DF02','DF03','DF04','DF06','MA01','MA02','MA05','MA07']

# Laestigkeits-Item umkodieren (5=gar nicht laestig statt 1=gar nicht laestig)
for name, prefix in verfahren.items():
    roh = f'{prefix}_02'
    neu = f'{prefix}_02_r'
    clean[neu] = 6 - clean[roh]

print("Umkodierung abgeschlossen. Beispiel SMS:")
print(clean[['A101_01','A101_02','A101_02_r','A101_03','A101_05','A101_06']].describe().round(2))

Umkodierung abgeschlossen. Beispiel SMS:
       A101_01  A101_02  A101_02_r  A101_03  A101_05  A101_06
count   163.00   163.00     163.00   163.00   163.00   163.00
mean      4.49     2.69       3.31     3.55     4.20     3.48
std       0.67     1.19       1.19     1.09     0.78     1.13
min       2.00     1.00       1.00     1.00     1.00     1.00
25%       4.00     2.00       2.00     3.00     4.00     3.00
50%       5.00     2.00       4.00     4.00     4.00     4.00
75%       5.00     4.00       4.00     4.00     5.00     4.00
max       5.00     5.00       5.00     5.00     5.00     5.00


## 4. Deskriptive Statistik pro Verfahren

Mittelwerte je Item, sowie die Rangfolge nach Nutzungsbereitschaft (`_05`), die für den Vergleich mit der Literatur zentral ist.

In [59]:
rows = []
for name, prefix in verfahren.items():
    rows.append({
        'Verfahren': name,
        'N (kennt Verfahren)': clean[f'{prefix}_05'].notna().sum(),
        'Einfachheit': clean[f'{prefix}_01'].mean(),
        'Nicht-Laestigkeit': clean[f'{prefix}_02_r'].mean(),
        'Vertrauen': clean[f'{prefix}_03'].mean(),
        'Bereitschaft': clean[f'{prefix}_05'].mean(),
        'Verstaendnis': clean[f'{prefix}_06'].mean(),
    })
desk = pd.DataFrame(rows).round(2)
desk = desk.sort_values('Bereitschaft', ascending=False).reset_index(drop=True)
desk

,Verfahren,N (kennt Verfahren),Einfachheit,Nicht-Laestigkeit,Vertrauen,Bereitschaft,Verstaendnis
0,SMS,163,4.49,3.31,3.55,4.20,3.48
1,Biometrie,163,4.49,3.99,3.84,4.18,3.65
2,AuthApp,146,3.94,2.84,4.14,4.01,3.27
3,FIDO2,100,3.43,2.99,3.75,3.29,2.88
4,HWToken,120,3.26,2.18,3.97,2.87,3.17


## 5. Gruppenvergleich: technisch vs. nicht-technisch (DF06)

Mann-Whitney-U als primärer Test (Standard in der Referenzliteratur, z.B. De Cristofaro et al. 2014, Krol et al. 2015, Zirjawi et al. 2015). Ergänzend ein t-Test als Robustheitscheck, da Likert-Daten laut Norman (2010) auch parametrisch robust auswertbar sind.

Effektstärke: Rangbiserialer Korrelationskoeffizient r für Mann-Whitney, Cohen's d für den t-Test.

In [60]:
def mannwhitney_mit_effektstaerke(gruppe1, gruppe2):
    """Mann-Whitney-U-Test mit rangbiserialem r als Effektstaerke."""
    g1 = gruppe1.dropna()
    g2 = gruppe2.dropna()
    if len(g1) < 2 or len(g2) < 2:
        return None
    u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    n1, n2 = len(g1), len(g2)
    r = 1 - (2*u) / (n1*n2)  # rangbiserial r
    return {'U': u, 'p': p, 'r': r, 'n_technisch': n1, 'n_nicht_technisch': n2}

def ttest_mit_cohens_d(gruppe1, gruppe2):
    g1 = gruppe1.dropna()
    g2 = gruppe2.dropna()
    if len(g1) < 2 or len(g2) < 2:
        return None
    t, p = stats.ttest_ind(g1, g2, equal_var=False)  # Welch's t-test
    n1, n2 = len(g1), len(g2)
    pooled_std = np.sqrt(((n1-1)*g1.std()**2 + (n2-1)*g2.std()**2) / (n1+n2-2))
    d = (g1.mean() - g2.mean()) / pooled_std
    return {'t': t, 'p': p, 'd': d}

tech = clean[clean['DF06'] == 1]
nontech = clean[clean['DF06'] == 2]
print(f"Technisch (DF06=1): n={len(tech)}  |  Nicht-technisch (DF06=2): n={len(nontech)}")

Technisch (DF06=1): n=68  |  Nicht-technisch (DF06=2): n=95


In [61]:
from IPython.display import display, Markdown

items = {
    '01':   'Einfachheit',
    '02_r': 'Nicht-Lästigkeit',
    '03':   'Vertrauen',
    '05':   'Nutzungsbereitschaft',
    '06':   'Verständnis',
}

tabellen = {}

for suffix, item_name in items.items():
    ergebnisse = []
    for name, prefix in verfahren.items():
        col = f'{prefix}_{suffix}'
        mw = mannwhitney_mit_effektstaerke(tech[col], nontech[col])
        tt = ttest_mit_cohens_d(tech[col], nontech[col])
        if mw is None:
            continue
        ergebnisse.append({
            'Verfahren': name,
            'M (technisch)': round(tech[col].mean(), 2),
            'M (nicht-techn.)': round(nontech[col].mean(), 2),
            'U': mw['U'],
            'p (Mann-Whitney)': round(mw['p'], 4),
            'r (Effektstärke)': round(mw['r'], 3),
            'p (t-Test)': round(tt['p'], 4) if tt else None,
            'd (Cohens d)': round(tt['d'], 3) if tt else None,
            'sig.': '*' if mw['p'] < 0.05 else '',
        })

    df_item = pd.DataFrame(ergebnisse)
    tabellen[item_name] = df_item

    display(Markdown(f"### {item_name}  (`_{suffix}`)"))
    display(df_item)

### Einfachheit  (`_01`)

,Verfahren,M (technisch),M (nicht-techn.),U,p (Mann-Whitney),r (Effektstärke),p (t-Test),d (Cohens d),sig.
0,SMS,4.47,4.51,3106.0,0.6347,0.038,0.7442,-0.052,
1,AuthApp,4.03,3.86,2915.5,0.2367,-0.108,0.2748,0.183,
2,Biometrie,4.54,4.45,3432.5,0.4274,-0.063,0.4526,0.118,
3,HWToken,3.26,3.26,1769.5,0.9481,0.007,0.9941,0.001,
4,FIDO2,3.36,3.51,1144.0,0.4693,0.081,0.4742,-0.144,


### Nicht-Lästigkeit  (`_02_r`)

,Verfahren,M (technisch),M (nicht-techn.),U,p (Mann-Whitney),r (Effektstärke),p (t-Test),d (Cohens d),sig.
0,SMS,3.50,3.18,3742.5,0.0722,-0.159,0.0891,0.272,
1,AuthApp,3.12,2.62,3238.5,0.0139,-0.230,0.0145,0.415,*
2,Biometrie,4.12,3.89,3601.5,0.1856,-0.115,0.1798,0.211,
3,HWToken,2.39,2.02,2056.5,0.1187,-0.154,0.0531,0.371,
4,FIDO2,2.96,3.02,1199.0,0.7420,0.037,0.7876,-0.054,


### Vertrauen  (`_03`)

,Verfahren,M (technisch),M (nicht-techn.),U,p (Mann-Whitney),r (Effektstärke),p (t-Test),d (Cohens d),sig.
0,SMS,3.79,3.38,3931.5,0.0146,-0.217,0.0169,0.387,*
1,AuthApp,4.40,3.93,3352.0,0.0024,-0.273,0.0016,0.526,*
2,Biometrie,3.97,3.75,3665.0,0.1266,-0.135,0.1884,0.212,
3,HWToken,4.22,3.76,2254.0,0.0089,-0.265,0.0106,0.474,*
4,FIDO2,4.00,3.47,1605.5,0.0096,-0.289,0.0076,0.550,*


### Nutzungsbereitschaft  (`_05`)

,Verfahren,M (technisch),M (nicht-techn.),U,p (Mann-Whitney),r (Effektstärke),p (t-Test),d (Cohens d),sig.
0,SMS,4.34,4.09,3706.5,0.0780,-0.148,0.0399,0.316,
1,AuthApp,4.23,3.83,3232.0,0.0122,-0.228,0.0061,0.451,*
2,Biometrie,4.22,4.15,3400.5,0.5372,-0.053,0.6279,0.077,
3,HWToken,2.98,2.77,1941.5,0.3889,-0.090,0.4018,0.154,
4,FIDO2,3.32,3.26,1278.5,0.8178,-0.026,0.7968,0.052,


### Verständnis  (`_06`)

,Verfahren,M (technisch),M (nicht-techn.),U,p (Mann-Whitney),r (Effektstärke),p (t-Test),d (Cohens d),sig.
0,SMS,3.85,3.21,4266.5,0.0003,-0.321,0.0003,0.588,*
1,AuthApp,3.71,2.93,3523.0,0.0003,-0.338,0.0009,0.539,*
2,Biometrie,3.99,3.41,4181.5,0.0008,-0.295,0.0009,0.526,*
3,HWToken,3.52,2.88,2345.5,0.0022,-0.316,0.0026,0.570,*
4,FIDO2,3.04,2.70,1430.5,0.1862,-0.149,0.1260,0.306,


**Interpretation der Effektstärke r:** ~0.10 klein, ~0.30 mittel, ~0.50 groß (Faustregel analog Cohen).
**Cohen's d:** ~0.20 klein, ~0.50 mittel, ~0.80 groß.

Stimmen p-Wert-Muster zwischen Mann-Whitney und t-Test überein, ist das Ergebnis robust gegenüber der Verteilungsannahme — relevant für die Norman-Diskussion in der Methodik.

## 6. Spearman-Korrelationen

In [10]:
print("=== Verstaendnis <-> Nutzungsbereitschaft (pro Verfahren) ===")
for name, prefix in verfahren.items():
    x = clean[f'{prefix}_06']
    y = clean[f'{prefix}_05']
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        continue
    rho, p = stats.spearmanr(x[mask], y[mask])
    print(f"  {name:<10} rho={rho:.3f}  p={p:.4f}  n={mask.sum()}")

=== Verstaendnis <-> Nutzungsbereitschaft (pro Verfahren) ===
  SMS        rho=0.243  p=0.0018  n=163
  AuthApp    rho=0.363  p=0.0000  n=146
  Biometrie  rho=0.363  p=0.0000  n=163
  HWToken    rho=0.370  p=0.0000  n=120
  FIDO2      rho=0.455  p=0.0000  n=100


In [11]:
print("=== DF04 (Selbsteinschaetzung) <-> DF06 (objektiver IT-Bezug) ===")
mask = clean['DF04'].notna() & clean['DF06'].notna()
rho, p = stats.spearmanr(clean.loc[mask,'DF04'], clean.loc[mask,'DF06'])
print(f"rho={rho:.3f}  p={p:.4f}  n={mask.sum()}")
print()
print("Kreuztabelle DF04 x DF06:")
print(pd.crosstab(clean['DF04'], clean['DF06']))

=== DF04 (Selbsteinschaetzung) <-> DF06 (objektiver IT-Bezug) ===
rho=-0.620  p=0.0000  n=163

Kreuztabelle DF04 x DF06:
DF06   1   2
DF04        
1      0   1
2      0  27
3     18  53
4     40  13
5     10   1


In [12]:
print("=== Alter (DF01) <-> Bereitschaft je Verfahren ===")
for name, prefix in verfahren.items():
    x = clean['DF01']
    y = clean[f'{prefix}_05']
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        continue
    rho, p = stats.spearmanr(x[mask], y[mask])
    print(f"  {name:<10} rho={rho:.3f}  p={p:.4f}  n={mask.sum()}")

=== Alter (DF01) <-> Bereitschaft je Verfahren ===
  SMS        rho=0.012  p=0.8750  n=163
  AuthApp    rho=0.112  p=0.1798  n=146
  Biometrie  rho=-0.024  p=0.7586  n=163
  HWToken    rho=0.226  p=0.0131  n=120
  FIDO2      rho=0.032  p=0.7535  n=100


## 7. Robustheitsprüfung: mit/ohne koreanische Teilstichprobe

Da Sprache und technischer Hintergrund in der Stichprobe ursprünglich konfundiert waren, wird hier geprüft, ob die Rangfolge der Verfahren und der Gruppenvergleich stabil bleiben.

In [13]:
ohne_kor = clean[clean['LANGUAGE'] != 'kor']
print(f"Gesamtstichprobe: n={len(clean)}  |  ohne Koreanisch: n={len(ohne_kor)}")
print()

vergleich_rows = []
for name, prefix in verfahren.items():
    col = f'{prefix}_05'
    vergleich_rows.append({
        'Verfahren': name,
        'M gesamt': clean[col].mean().round(2),
        'M ohne KOR': ohne_kor[col].mean().round(2),
    })
pd.DataFrame(vergleich_rows).sort_values('M gesamt', ascending=False)

Gesamtstichprobe: n=163  |  ohne Koreanisch: n=90



,Verfahren,M gesamt,M ohne KOR
0,SMS,4.20,4.39
2,Biometrie,4.18,4.40
1,AuthApp,4.01,4.22
4,FIDO2,3.29,3.63
3,HWToken,2.87,3.05


In [14]:
print("Gruppenvergleich technisch/nicht-technisch NUR innerhalb Koreanisch:")
kor = clean[clean['LANGUAGE'] == 'kor']
kor_tech = kor[kor['DF06'] == 1]
kor_nontech = kor[kor['DF06'] == 2]
print(f"n technisch (KOR)={len(kor_tech)}  |  n nicht-technisch (KOR)={len(kor_nontech)}")
print()

for name, prefix in verfahren.items():
    col = f'{prefix}_05'
    mw = mannwhitney_mit_effektstaerke(kor_tech[col], kor_nontech[col])
    if mw is None:
        continue
    print(f"  {name:<10} p={mw['p']:.4f}  r={mw['r']:.3f}")

Gruppenvergleich technisch/nicht-technisch NUR innerhalb Koreanisch:
n technisch (KOR)=21  |  n nicht-technisch (KOR)=52

  SMS        p=0.1159  r=-0.204
  AuthApp    p=0.0583  r=-0.282
  Biometrie  p=0.6883  r=0.057
  HWToken    p=0.6832  r=0.062
  FIDO2      p=0.5289  r=0.118


## 8. Allgemeine Aussagen zu MFA (AC01/AC05-Batterien, ohne Attention-Check-Item)

In [15]:
for c in ['AC01_01','AC01_02','AC01_04','AC05_01','AC05_03']:
    clean[c] = pd.to_numeric(clean[c], errors='coerce')

print("Sicherheitsbewusstsein (AC01, Anfang der Umfrage):")
print(f"  Sicherheit wichtig:        M={clean['AC01_01'].mean():.2f}")
print(f"  Online-Sicherheit aufwaendig: M={clean['AC01_02'].mean():.2f}")
print(f"  Fuehle mich informiert:    M={clean['AC01_04'].mean():.2f}")
print()
print("Gesamteinstellung (AC05, Ende der Umfrage):")
print(f"  Wuerde MFA empfehlen:      M={clean['AC05_01'].mean():.2f}")
print(f"  Fuehle mich sicherer:      M={clean['AC05_03'].mean():.2f}")

Sicherheitsbewusstsein (AC01, Anfang der Umfrage):
  Sicherheit wichtig:        M=4.65
  Online-Sicherheit aufwaendig: M=3.47
  Fuehle mich informiert:    M=3.40

Gesamteinstellung (AC05, Ende der Umfrage):
  Wuerde MFA empfehlen:      M=4.06
  Fuehle mich sicherer:      M=4.00


## 9. Zusammenfassung — Kopiervorlage für den Ergebnisteil

In [62]:
print("="*60)
print("STICHPROBE")
print("="*60)
print(f"N (bereinigt) = {len(clean)}")
print(f"  davon Prolific: {(clean['quelle']=='Prolific').sum()}, Umfeld: {(clean['quelle']=='Umfeld').sum()}")
print(f"  IT-Bezug: Ja={( clean['DF06']==1).sum()}, Nein={(clean['DF06']==2).sum()}")
print(f"  Sprachen: {dict(clean['LANGUAGE'].value_counts())}")
print()
print("="*60)
print("RANGFOLGE NACH NUTZUNGSBEREITSCHAFT")
print("="*60)
print(desk[['Verfahren','Bereitschaft']].to_string(index=False))
print()
print("="*60)
print("GRUPPENVERGLEICH (signifikant bei p<.05)")
print("="*60)
sig = gruppenvergleich[gruppenvergleich['p (Mann-Whitney)'] < 0.05]
if len(sig):
    print(sig.to_string(index=False))
else:
    print("Keine signifikanten Unterschiede gefunden.")

STICHPROBE
N (bereinigt) = 163
  davon Prolific: 80, Umfeld: 83
  IT-Bezug: Ja=68, Nein=95
  Sprachen: {'kor': np.int64(73), 'ger': np.int64(61), 'eng': np.int64(29)}

RANGFOLGE NACH NUTZUNGSBEREITSCHAFT
Verfahren  Bereitschaft
      SMS          4.20
Biometrie          4.18
  AuthApp          4.01
    FIDO2          3.29
  HWToken          2.87

GRUPPENVERGLEICH (signifikant bei p<.05)
Verfahren  M (technisch)  M (nicht-techn.)      U  p (Mann-Whitney)  r (Effektstaerke)  p (t-Test)  d (Cohens d)
      SMS           3.79              3.38 3931.5            0.0146             -0.217      0.0169         0.387
  AuthApp           4.40              3.93 3352.0            0.0024             -0.273      0.0016         0.526
  HWToken           4.22              3.76 2254.0            0.0089             -0.265      0.0106         0.474
    FIDO2           4.00              3.47 1605.5            0.0096             -0.289      0.0076         0.550


In [66]:
from scipy import stats
from IPython.display import display, Markdown

tech = clean[clean['DF06'] == 1]
nontech = clean[clean['DF06'] == 2]

# ---------------------------------------------------------------
# 1. Allgemeine Aussagen nach IT-Bezug
# ---------------------------------------------------------------
allgemein = {
    'AC01_01': 'Sicherheit im Internet ist mir wichtig',
    'AC01_02': 'Online-Sicherheit zu managen ist aufwändig',
    'AC01_04': 'Fühle mich beim Thema gut informiert',
    'AC05_01': 'Würde MFA weiterempfehlen',
    'AC05_03': 'Fühle mich durch MFA sicherer',
}

zeilen = []
for col, label in allgemein.items():
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
    clean[col] = clean[col].where(clean[col] >= 0)
    g1 = clean[clean['DF06'] == 1][col].dropna()
    g2 = clean[clean['DF06'] == 2][col].dropna()
    u, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    r = 1 - (2*u)/(len(g1)*len(g2))
    zeilen.append({
        'Aussage': label,
        'M (technisch)': round(g1.mean(), 2),
        'M (nicht-techn.)': round(g2.mean(), 2),
        'M gesamt': round(clean[col].mean(), 2),
        'p': round(p, 4),
        'r': round(r, 3),
        'sig.': '*' if p < 0.05 else '',
    })

display(Markdown("### Allgemeine Aussagen nach IT-Bezug"))
display(pd.DataFrame(zeilen))

### Allgemeine Aussagen nach IT-Bezug

,Aussage,M (technisch),M (nicht-techn.),M gesamt,p,r,sig.
0,Sicherheit im Internet ist mir wichtig,4.75,4.58,4.65,0.0433,-0.153,*
1,Online-Sicherheit zu managen ist aufwändig,3.24,3.76,3.54,0.0033,0.258,*
2,Fühle mich beim Thema gut informiert,3.74,3.16,3.40,0.0000,-0.351,*
3,Würde MFA weiterempfehlen,4.37,3.83,4.06,0.0003,-0.316,*
4,Fühle mich durch MFA sicherer,4.18,3.87,4.00,0.0190,-0.202,*


In [64]:
# ---------------------------------------------------------------
# 2. Präferenzfragen nach IT-Bezug
# ---------------------------------------------------------------
labels_pref = {1:'SMS-Code', 2:'Authenticator-App', 3:'Biometrie',
               4:'Hardware-Token', 5:'FIDO2/WebAuthn', 6:'keines / keine Abneigung'}

for col, titel in [('MA05', 'Bevorzugtes Verfahren'),
                   ('MA07', 'Am wenigsten gewünschtes Verfahren')]:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
    clean[col] = clean[col].where(clean[col] >= 0)

    tab = pd.DataFrame({
        'gesamt':          clean[col].value_counts(normalize=True) * 100,
        'technisch':       clean[clean['DF06']==1][col].value_counts(normalize=True) * 100,
        'nicht-technisch': clean[clean['DF06']==2][col].value_counts(normalize=True) * 100,
    }).round(1)
    tab.index = [labels_pref.get(i, i) for i in tab.index]
    tab = tab.sort_values('gesamt', ascending=False)

    display(Markdown(f"### {titel} (Angaben in %)"))
    display(tab)

    # Chi-Quadrat-Test auf Unabhängigkeit
    kreuz = pd.crosstab(clean[col], clean['DF06'])
    chi2, p, dof, exp = stats.chi2_contingency(kreuz)
    print(f"Chi-Quadrat-Test: χ² = {chi2:.2f}, df = {dof}, p = {p:.4f}")
    if (exp < 5).sum() > 0:
        print(f"Hinweis: {(exp < 5).sum()} Zellen mit erwarteter Häufigkeit < 5 "
              f"→ Testergebnis nur eingeschränkt belastbar")
    print()

### Bevorzugtes Verfahren (Angaben in %)

,gesamt,technisch,nicht-technisch
Biometrie,51.5,44.1,56.8
SMS-Code,23.3,17.6,27.4
Authenticator-App,19.6,30.9,11.6
Hardware-Token,2.5,2.9,2.1
FIDO2/WebAuthn,2.5,4.4,1.1
keines / keine Abneigung,0.6,NaN,1.1


Chi-Quadrat-Test: χ² = 13.03, df = 5, p = 0.0231
Hinweis: 6 Zellen mit erwarteter Häufigkeit < 5 → Testergebnis nur eingeschränkt belastbar



### Am wenigsten gewünschtes Verfahren (Angaben in %)

,gesamt,technisch,nicht-technisch
Hardware-Token,42.9,44.1,42.1
FIDO2/WebAuthn,19.6,26.5,14.7
keines / keine Abneigung,18.4,11.8,23.2
Biometrie,9.8,10.3,9.5
Authenticator-App,5.5,2.9,7.4
SMS-Code,3.7,4.4,3.2


Chi-Quadrat-Test: χ² = 7.22, df = 5, p = 0.2051
Hinweis: 3 Zellen mit erwarteter Häufigkeit < 5 → Testergebnis nur eingeschränkt belastbar



In [67]:
# Informiertheitsgefühl: Welche Vorwissens-Variable erklärt mehr?
print("=== AC01_04 (Fühle mich informiert) ↔ Vorwissen ===")
for var, label in [('DF04', 'Selbsteinschätzung'), ('DF06', 'objektiver IT-Bezug')]:
    m = clean['AC01_04'].notna() & clean[var].notna()
    rho, p = stats.spearmanr(clean.loc[m, 'AC01_04'], clean.loc[m, var])
    print(f"  {label:<22} ρ = {rho:+.3f}, p = {p:.4f}, n = {m.sum()}")

print()
print("Hinweis zur Lesart: DF04 ist aufsteigend kodiert (1 = keine Kenntnisse),")
print("DF06 absteigend (1 = IT-Bezug ja). Die Vorzeichen sind daher gegenläufig,")
print("inhaltlich zeigen beide in dieselbe Richtung.")

=== AC01_04 (Fühle mich informiert) ↔ Vorwissen ===
  Selbsteinschätzung     ρ = +0.436, p = 0.0000, n = 163
  objektiver IT-Bezug    ρ = -0.319, p = 0.0000, n = 163

Hinweis zur Lesart: DF04 ist aufsteigend kodiert (1 = keine Kenntnisse),
DF06 absteigend (1 = IT-Bezug ja). Die Vorzeichen sind daher gegenläufig,
inhaltlich zeigen beide in dieselbe Richtung.
